# 02 · Download validation data for the season dates

Independent dates to check the Sentinel-2 season dates of `04` against, from the **MeteoSwiss phenology network** (open data, no account): European larch *needle colouring 50 %* and *needle drop 50 %*, observed by eye at stations near the AOI. Colouring 50 % corresponds to the yellowing peak / midpoint, needle drop 50 % to the end of the season. The stations lie lower than most of the AOI, so `04` compares year-to-year anomalies, not absolute dates.

- **Settings:** `config.py` (`VALIDATION_RADIUS_KM`, `PHENO_PARAMS`, years)
- **Reads:** `cache/<AOI>_zones/zones.tif` (from `01_download_sentinel2`)
- **Writes** to `cache/validation/<AOI>/`:
  - `meteoswiss_larch.csv`: one row per station, year and phase
  - `meteoswiss_stations.csv`: the stations used
- **Next:** `04_phenology_metrics` (section 7 compares both methods with these data)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray
from pyproj import Transformer

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import collection_asset, download, stac_items

PHENO_COLLECTION = "ch.meteoschweiz.ogd-phenology"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

zones_tif = rioxarray.open_rasterio(CACHE_DIR / "zones.tif", masked=True)
x0, y0, x1, y1 = zones_tif.rio.bounds()
to_wgs84 = Transformer.from_crs(zones_tif.rio.crs, "EPSG:4326", always_xy=True)
AOI_LON, AOI_LAT = to_wgs84.transform((x0 + x1) / 2, (y0 + y1) / 2)
print(f"AOI: {AOI_NAME} | centre {AOI_LAT:.3f} N, {AOI_LON:.3f} E | writes to {VALIDATION_DIR}")

## MeteoSwiss phenology observations (European larch)

The Swiss Phenology Network (about 160 stations) records the date when 50 % of the larch needles at a site are coloured (`mlard94d`) or have fallen (`mlard95d`). Data: MeteoSwiss open data, collection `ch.meteoschweiz.ogd-phenology` (one CSV per station, one column per phase, dates as YYYYMMDD).

Stations within `VALIDATION_RADIUS_KM` of the AOI centre that observe one of `PHENO_PARAMS` are kept.

In [ ]:
def read_meta(name):
    path = download(collection_asset(PHENO_COLLECTION, name), VALIDATION_DIR / "meteoswiss" / name)
    return pd.read_csv(path, sep=";", encoding="cp1252")


stations = read_meta("ogd-phenology_meta_stations.csv").set_index("station_abbr")
inventory = read_meta("ogd-phenology_meta_datainventory.csv")
observing = inventory[inventory.parameter_shortname.isin(PHENO_PARAMS)].station_abbr.unique()
lat, lon = stations.station_coordinates_wgs84_lat, stations.station_coordinates_wgs84_lon
stations["distance_km"] = 6371 * np.hypot(np.radians(lat - AOI_LAT), np.radians(lon - AOI_LON) * np.cos(np.radians(AOI_LAT)))
near = stations.loc[observing].query("distance_km <= @VALIDATION_RADIUS_KM").sort_values("distance_km")
near = near[["station_name", "station_height_masl", "distance_km", "station_coordinates_wgs84_lat",
             "station_coordinates_wgs84_lon"]].rename(columns=lambda c: c.replace("station_", "").replace("coordinates_wgs84_", ""))
print(f"{len(near)} stations within {VALIDATION_RADIUS_KM} km observe larch needle colouring or drop")

# One CSV per station (STAC item id = lower-case abbreviation)
assets = {item["id"].upper(): next(iter(item["assets"].values()))["href"] for item in stac_items(PHENO_COLLECTION)}
rows = []
for abbr in near.index:
    path = download(assets[abbr], VALIDATION_DIR / "meteoswiss" / f"{abbr.lower()}_y.csv")
    data = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
    year = pd.to_datetime(data.reference_timestamp, format="%d.%m.%Y %H:%M").dt.year
    for param, phase in PHENO_PARAMS.items():
        if param not in data:
            continue
        when = pd.to_datetime(data[param], format="%Y%m%d", errors="coerce")
        ok = when.notna()
        rows.append(pd.DataFrame({"station": abbr, "year": year[ok], "param": param, "phase": phase,
                                  "date": when[ok].dt.date, "doy": when[ok].dt.dayofyear}))
larch_obs = pd.concat(rows, ignore_index=True).join(near[["name", "height_masl", "distance_km"]], on="station")
larch_obs = larch_obs[larch_obs.year.between(FIRST_YEAR - 10, LAST_YEAR)]   # some years before for the station means
near = near.join(larch_obs[larch_obs.year >= FIRST_YEAR].pivot_table(index="station", columns="param", values="year",
                                                                    aggfunc="nunique").add_prefix("years_"))
near = near.dropna(how="all", subset=[c for c in near if c.startswith("years_")])   # no observations since FIRST_YEAR
display(near.round(1))

larch_obs.to_csv(VALIDATION_DIR / "meteoswiss_larch.csv", index=False)
near.to_csv(VALIDATION_DIR / "meteoswiss_stations.csv")

fig, axes = plt.subplots(1, len(PHENO_PARAMS), figsize=(12, 4), sharey=True)
for ax, (param, phase) in zip(axes, PHENO_PARAMS.items()):
    for abbr, g in larch_obs[(larch_obs.param == param) & (larch_obs.year >= FIRST_YEAR)].groupby("station"):
        ax.plot(g.year, g.doy, "o-", ms=4, label=f"{abbr} ({near.height_masl[abbr]:.0f} m)")
    ax.set_title(f"Larch {phase}")
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("day of year")
axes[-1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()